In [1]:

import os
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document

In [3]:
# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

docs = [doc1, doc2, doc3, doc4, doc5]

In [4]:
embedding = HuggingFaceEndpointEmbeddings(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

f:\PANTA\due\LangChain\langchain_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
vector_store = FAISS.from_documents(docs, embedding)

In [6]:
# Save to disk
vector_store.save_local("faiss_ipl_index")

In [7]:
# Load Vector Store (if needed)
# vector_store = FAISS.load_local("faiss_ipl_index", embedding)

In [8]:
# Cell 7: Similarity Search
results = vector_store.similarity_search("Who among these are a bowler?", k=2)
for res in results:
    print(f"\nMatched: {res.page_content}\nMetadata: {res.metadata}")


Matched: Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.
Metadata: {'team': 'Mumbai Indians'}

Matched: Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.
Metadata: {'team': 'Mumbai Indians'}


In [9]:
# Cell 8: Similarity Search with Scores
results_with_score = vector_store.similarity_search_with_score("Who among these are a bowler?", k=2)
for doc, score in results_with_score:
    print(f"\nMatched: {doc.page_content}\nScore: {score:.4f}\nMetadata: {doc.metadata}")



Matched: Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.
Score: 0.9694
Metadata: {'team': 'Mumbai Indians'}

Matched: Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.
Score: 1.1493
Metadata: {'team': 'Mumbai Indians'}


In [10]:
# Cell 10: Update Document (FAISS requires rebuild)
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

In [11]:
# Replace old doc with updated one
updated_docs = [updated_doc1, doc2, doc3, doc4, doc5]
vector_store_updated = FAISS.from_documents(updated_docs, embedding)
vector_store_updated.save_local("faiss_ipl_index_updated")

In [12]:
# Cell 11: Delete a Document (FAISS requires rebuild)
# Delete Bumrah's doc (doc4)
remaining_docs = [doc for doc in updated_docs if "Bumrah" not in doc.page_content]
vector_store_pruned = FAISS.from_documents(remaining_docs, embedding)
vector_store_pruned.save_local("faiss_ipl_index_pruned")